# A2.5 · Delegation that survives audit

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.4 · The NHI governance gap](https://spbreed.github.io/cyber-commons/lessons/A2.4.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, RFC 8693 |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Now we build the thing A2.1 pointed at and A2.3 proved we need:
**on-behalf-of delegation**, standardised as [RFC 8693 OAuth 2.0 Token
Exchange](https://datatracker.ietf.org/doc/html/rfc8693).

The idea is small. An actor presents a token it holds and asks for a new one for
a different actor. The issuer returns a token with:

- **`sub`** — unchanged. The action is still *for* the original principal.
- **`actor`** — the new holder.
- **`act`** — a nested claim recording who presented the token, and who
  presented it to *them*, all the way back.

Two rules make the result auditable, and both must hold:

1. **Subset of what was presented.** You cannot hand on authority you were not
   given.
2. **Within the new actor's own ceiling.** You cannot hand on authority the
   recipient may never hold, even if the caller offered it. (This is A1.3's
   ceiling, doing its second job.)

Drop either rule and the chain still *looks* correct — every token parses, every
call succeeds — which is precisely why this needs a test rather than a review.

## 2 · Demo — a three-hop chain that narrows at every step

Dana asks for a fix. The orchestrator delegates to a patch agent, which delegates to a deploy agent. Watch the scopes shrink.

In [ ]:
import hashlib, json, time
from dataclasses import dataclass, field

# A1.3's ceilings: what each actor may EVER hold.
CEILINGS = {
    "dana@corp":    {"repo:read", "repo:write", "deploy:prod", "secrets:read"},
    "orchestrator": {"repo:read", "repo:write", "deploy:prod"},
    "patch-agent":  {"repo:read", "repo:write"},
    "deploy-agent": {"repo:read", "deploy:prod"},
    "triage-agent": {"repo:read"},
}

class DelegationError(Exception):
    """Refusing to widen is the feature."""

@dataclass
class Token:
    sub: str
    actor: str
    scopes: set
    act: dict = None
    issued: float = field(default_factory=time.time)
    ttl: float = 300

    @property
    def expired(self): return time.time() - self.issued > self.ttl

    def chain(self):
        out, node = [], self.act
        while node:
            out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub:
            c.insert(0, self.sub)
        return c

    def fingerprint(self):
        blob = json.dumps({"sub": self.sub, "actor": self.actor,
                           "scopes": sorted(self.scopes), "act": self.act},
                          sort_keys=True)
        return hashlib.sha256(blob.encode()).hexdigest()[:12]

    def describe(self):
        return (f"{' → '.join(self.chain()):48s}\n"
                f"      scopes {sorted(self.scopes)}   fp={self.fingerprint()}")

def mint(principal, scopes=None):
    ceiling = CEILINGS[principal]
    want = set(scopes) if scopes else set(ceiling)
    if not want <= ceiling:
        raise DelegationError(f"{principal} cannot hold {sorted(want - ceiling)}")
    return Token(sub=principal, actor=principal, scopes=want)

def exchange(presented, new_actor, scopes):
    """One RFC 8693 hop. Both narrowing rules live here and nowhere else."""
    if presented.expired:
        raise DelegationError("presented token has expired")
    scopes = set(scopes)
    if not scopes <= presented.scopes:                       # rule 1
        raise DelegationError(
            f"widening refused: {sorted(scopes - presented.scopes)} is not in the "
            f"presented token {sorted(presented.scopes)}")
    ceiling = CEILINGS.get(new_actor, set())
    if not scopes <= ceiling:                                # rule 2
        raise DelegationError(
            f"widening refused: {new_actor} may never hold "
            f"{sorted(scopes - ceiling)} (ceiling {sorted(ceiling)})")
    return Token(sub=presented.sub, actor=new_actor, scopes=scopes,
                 act={"actor": presented.actor, "act": presented.act},
                 ttl=min(presented.ttl, 300))

dana  = mint("dana@corp", {"repo:read", "repo:write", "deploy:prod"})
orch  = exchange(dana, "orchestrator", {"repo:read", "repo:write", "deploy:prod"})
patch = exchange(orch, "patch-agent",  {"repo:read", "repo:write"})
ship  = exchange(patch, "deploy-agent", {"repo:read"})

for t in (dana, orch, patch, ship):
    print(t.describe())

## 3 · Demo — the resource server can finally answer the question

This is the payoff. GitHub (or any downstream) receives the last token and can record a truthful, complete line.

In [ ]:
def resource_server(token, required_scope):
    if token.expired:
        return {"allowed": False, "why": "token expired"}
    if required_scope not in token.scopes:
        return {"allowed": False,
                "why": f"needs {required_scope}, holds {sorted(token.scopes)}"}
    return {"allowed": True,
            "audit": f"{token.actor} performed {required_scope} on behalf of "
                     f"{token.sub} via {' → '.join(token.chain()[1:-1]) or 'direct'}",
            "chain": token.chain()}

for tok, scope in ((patch, "repo:write"), (ship, "repo:write"), (ship, "repo:read")):
    r = resource_server(tok, scope)
    print(f"{tok.actor:14s} wants {scope:12s} → {'ALLOW' if r['allowed'] else 'DENY '}")
    print(f"   {r.get('audit') or r['why']}")

## 4 · Where it breaks — three ways, all refused

Now the attacks. Each of these is something a real integration will attempt, usually by accident.

In [ ]:
attacks = [
 ("widen beyond the presented token",
  lambda: exchange(ship, "deploy-agent", {"deploy:prod"})),
 ("widen beyond the actor's own ceiling",
  lambda: exchange(dana, "triage-agent", {"repo:write"})),
 ("replay an expired token",
  lambda: exchange(Token("dana@corp", "patch-agent", {"repo:write"}, ttl=-1),
                   "deploy-agent", {"repo:write"})),
]
for name, fn in attacks:
    try:
        fn()
        print(f"GRANTED  {name}   ← this must not happen")
    except DelegationError as e:
        print(f"REFUSED  {name}\n         {e}")

## 5 · The anti-pattern, for contrast

Impersonation produces a token that works perfectly and destroys the audit trail — the A2.3 failure, now visible next to the correct version.

In [ ]:
def impersonate(principal, actor, scopes):
    """No act claim. The agent simply becomes the human."""
    return Token(sub=principal, actor=principal, scopes=set(scopes), act=None)

bad = impersonate("dana@corp", "patch-agent", {"repo:write"})
print("delegated    :", " → ".join(patch.chain()))
print("impersonated :", " → ".join(bad.chain()), "  ← the agent is invisible")
print("\nresource server sees:")
print("   delegated    :", resource_server(patch, "repo:write")["audit"])
print("   impersonated :", resource_server(bad, "repo:write")["audit"])

In [ ]:
# Verify: property-test the invariant over random chains.
import random
random.seed(11)
actors = [a for a in CEILINGS if a != "dana@corp"]
violations, built = 0, 0
for _ in range(1500):
    tok = mint("dana@corp")
    for _ in range(random.randint(1, 4)):
        nxt = random.choice(actors)
        want = set(random.sample(sorted(tok.scopes),
                                 k=random.randint(0, len(tok.scopes))))
        try:
            new = exchange(tok, nxt, want)
        except DelegationError:
            continue
        if not new.scopes <= tok.scopes or not new.scopes <= CEILINGS[nxt]:
            violations += 1
        tok = new; built += 1
print(f"{built} successful hops across 1500 random chains — widening violations: {violations}")
assert violations == 0
print("Invariant holds: authority can only shrink, on every path.")

## What you just proved

Four tokens print with strictly narrowing scopes and readable chains ending in `dana@corp → orchestrator → patch-agent → deploy-agent`. The resource server allows `patch-agent` a write, denies `deploy-agent` the same write, and produces a truthful audit line naming both the actor and the principal. All three attacks are refused with the rule that refused them. The impersonated token's chain contains only `dana@corp`. The property test reports zero widening violations.

## Your turn

Run these same four scenarios against real Keycloak with token exchange enabled. The properties should hold identically — and if your realm allows the second one (widening past the actor's ceiling), that is a live finding, because Keycloak will happily issue it if the client is configured permissively.

---

**Next → [A2.6 · The agentic gateway](https://spbreed.github.io/cyber-commons/lessons/A2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*